# Instamart Customer Intelligence & Support Copilot
## Notebook 2 of 2 — Part C (Support Ticket Triage) and Part D (Policy RAG Assistant & Refund Agent)

| Part | What this notebook does | Main outputs |
|---|---|---|
| **C** | Guardrails, PII redaction, prompts, retries, output validation, zero-/few-shot evaluation, token budget, routing using the Part B segments | `outputs/triage_budget_demo.csv`, `outputs/triage_results.csv` |
| **D** | Chunk policies, TF-IDF retrieval, grounded policy assistant, evaluation, refund tools, bounded and resumable refund agent | `outputs/refund_ledger.jsonl`, `outputs/agent_runs.jsonl`, `state/` |

All language-model calls go through the supplied offline `mock_llm.py` (no API key, no network).

**Run Notebook 1 first.** This notebook needs `data/instamart.db` (Part A) and `outputs/customer_segments.csv` (Part B). Then run every cell here from top to bottom.

## 0. Project setup and prerequisites

In [2]:
import os
import sys
import subprocess
from pathlib import Path

# Find the repository root (the folder that holds generate_data.py), so the notebook
# works whether it is saved in the repo root or in a notebooks/ sub-folder.
ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "generate.py").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("generate.py not found - open this notebook inside the project folder.")

os.chdir(ROOT)                 # all paths below (data/raw/, sql/, outputs/ ...) are relative to ROOT
sys.path.insert(0, str(ROOT))  # so that `import mock_llm` works
print("Project root:", ROOT)
print("Root files  :", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

Project root: C:\Users\Lavanya Ramakrishnan\OneDrive\Desktop\Capstone project
Root files  : ['.gitignore', 'generate.py', 'mock_llm.py', 'requirements.txt', 'run_project.py']


In [3]:
required = ["data/raw/tickets.csv", "data/raw/policy_eval.csv", "data/raw/agent_scenarios.csv",
            "data/policies", "data/instamart.db", "outputs/customer_segments.csv", "mock_llm.py"]
missing = [p for p in required if not (ROOT / p).exists()]
if missing:
    raise FileNotFoundError(f"Missing {missing}. Run Notebook 1 (Part A and Part B) first.")
print("All prerequisites found.")

import pandas as pd
import numpy as np
import re
import json
import random
import time
import uuid
import sqlite3

import mock_llm

All prerequisites found.


# PART C — Support Ticket Triage with Guardrails

## Task 0. Configuration and data
`keep_default_na=False` keeps empty ticket text as an empty string (instead of NaN) so the empty-ticket guardrail can detect it explicitly.

In [4]:
# Configuration
MAX_INPUT_TOKENS = 400
BASE_DELAY = 0.05
MAX_RETRIES = 3
TOKEN_BUDGET = 100000

# Retry counters (reset for each run_pipeline call)
RETRY_FAILURE_COUNT = 0
PERMANENT_FAILURE_COUNT = 0

ALLOWED_CATEGORIES = {
    "billing",
    "delivery",
    "technical",
    "fraud"
}

# Load tickets
tickets = pd.read_csv(
    "data/raw/tickets.csv",
    keep_default_na=False
)

print("\n========== TICKET DATA ==========")
print("Shape:", tickets.shape)
print("Columns:", tickets.columns.tolist())
print(tickets.head())


========== TICKET DATA ==========
Shape: (164, 5)
Columns: ['ticket_id', 'customer_id', 'order_id', 'text', 'true_category']
  ticket_id customer_id    order_id  \
0     T0024       C0486  ORD-105035   
1     T0073       C0080  ORD-100816   
2     T0095       C1087  ORD-111156   
3     T0159       C1172               
4     T0080       C0910  ORD-109516   

                                                text true_category  
0  There is an unauthorized transaction on my car...         fraud  
1  The invoice for ORD-100816 shows a wrong amoun...       billing  
2  The payment page shows an error and the app fr...     technical  
3                                                             none  
4  I was charged for order ORD-109516 that I neve...         fraud  


## Task 1. Input guardrails
Empty tickets and over-long tickets (more than 400 estimated tokens) must **not** be sent to the model. Tokens are estimated with the project-wide `mock_llm.est_tokens`.

In [5]:
# ==========================================
# TASK 1: INPUT GUARDRAILS
# ==========================================

# Empty ticket check
tickets["text"] = tickets["text"].astype(str)

tickets["is_empty"] = tickets["text"].str.strip().eq("")

# Token estimation using mock_llm
tickets["estimated_tokens"] = tickets["text"].apply(
    mock_llm.est_tokens
)

# Identify long tickets
tickets["is_long"] = (
    tickets["estimated_tokens"] > MAX_INPUT_TOKENS
)

print("\n========== INPUT GUARDRAILS ==========")
print("Empty tickets:", tickets["is_empty"].sum())
print("Long tickets:", tickets["is_long"].sum())

print("\nLong ticket token estimates:")
print(
    tickets.loc[
        tickets["is_long"],
        ["ticket_id", "estimated_tokens"]
    ]
)


========== INPUT GUARDRAILS ==========
Empty tickets: 4
Long tickets: 4

Long ticket token estimates:
    ticket_id  estimated_tokens
10      T0163               734
78      T0162               734
143     T0164               734
158     T0161               734


## Task 2. PII redaction
Emails, phone numbers and order IDs are replaced by `[EMAIL]`, `[PHONE]`, `[ORDER_ID]` **before** the text reaches the model. The order matters (email → phone → order ID), and the check at the end asserts that nothing is left behind.

In [6]:
# ==========================================
# TASK 2: PII REDACTION
# ==========================================

# Exact patterns and placeholders
EMAIL_PATTERN = r"[\w.+-]+@[\w-]+\.[\w.]+"
PHONE_PATTERN = r"\b[6-9]\d{9}\b"
ORDER_PATTERN = r"ORD-\d{6}"

def redact_pii(text):
    email_count = len(re.findall(EMAIL_PATTERN, text))
    phone_count = len(re.findall(PHONE_PATTERN, text))
    order_count = len(re.findall(ORDER_PATTERN, text))

    text = re.sub(EMAIL_PATTERN, "[EMAIL]", text)
    text = re.sub(PHONE_PATTERN, "[PHONE]", text)
    text = re.sub(ORDER_PATTERN, "[ORDER_ID]", text)

    return text, email_count, phone_count, order_count

# Apply redaction to all tickets
redaction_results = tickets["text"].apply(redact_pii)

tickets["redacted_text"] = redaction_results.apply(lambda x: x[0])
tickets["email_count"] = redaction_results.apply(lambda x: x[1])
tickets["phone_count"] = redaction_results.apply(lambda x: x[2])
tickets["order_count"] = redaction_results.apply(lambda x: x[3])

print("\n========== REDACTION RESULTS ==========")
print("Emails:", tickets["email_count"].sum())
print("Phones:", tickets["phone_count"].sum())
print("Order IDs:", tickets["order_count"].sum())
print(
    "Total substitutions:",
    tickets[["email_count", "phone_count", "order_count"]].sum().sum()
)

# Verify no PII remains after redaction
remaining_emails = tickets["redacted_text"].apply(
    lambda x: len(re.findall(EMAIL_PATTERN, x))
).sum()

remaining_phones = tickets["redacted_text"].apply(
    lambda x: len(re.findall(PHONE_PATTERN, x))
).sum()

remaining_orders = tickets["redacted_text"].apply(
    lambda x: len(re.findall(ORDER_PATTERN, x))
).sum()

print("\nRemaining emails:", remaining_emails)
print("Remaining phones:", remaining_phones)
print("Remaining order IDs:", remaining_orders)

assert remaining_emails == 0
assert remaining_phones == 0
assert remaining_orders == 0


========== REDACTION RESULTS ==========
Emails: 55
Phones: 91
Order IDs: 132
Total substitutions: 278

Remaining emails: 0
Remaining phones: 0
Remaining order IDs: 0


## Task 3. System prompts: zero-shot and few-shot
Both prompts contain the word `CLASSIFY` (it selects the classifier behaviour inside `mock_llm`). The few-shot prompt adds an `EXAMPLES:` block.

In [7]:
ZERO_SHOT_PROMPT = """
CLASSIFY the customer support ticket into exactly one category.

Allowed categories:
- billing
- delivery
- technical
- fraud

Return JSON only in this format:
{"category": "...", "reason": "..."}
"""

FEW_SHOT_PROMPT = """
CLASSIFY the customer support ticket into exactly one category.

Allowed categories:
- billing
- delivery
- technical
- fraud

EXAMPLES:
Ticket: I was charged twice for my order.
{"category": "billing", "reason": "Duplicate charge"}

Ticket: My package has not arrived.
{"category": "delivery", "reason": "Package not received"}

Ticket: Someone used my card without permission.
{"category": "fraud", "reason": "Unauthorized transaction"}

Return JSON only in this format:
{"category": "...", "reason": "..."}
"""

print("\n========== PROMPT VERIFICATION ==========")
print("Zero-shot contains CLASSIFY:", "CLASSIFY" in ZERO_SHOT_PROMPT)
print("Zero-shot contains EXAMPLES:", "EXAMPLES:" in ZERO_SHOT_PROMPT)
print("Few-shot contains CLASSIFY:", "CLASSIFY" in FEW_SHOT_PROMPT)
print("Few-shot contains EXAMPLES:", "EXAMPLES:" in FEW_SHOT_PROMPT)


========== PROMPT VERIFICATION ==========
Zero-shot contains CLASSIFY: True
Zero-shot contains EXAMPLES: False
Few-shot contains CLASSIFY: True
Few-shot contains EXAMPLES: True


## Task 4. Retry with exponential backoff and jitter
The mock model fails every 9th call with `TransientLLMError`. Each failed call is retried up to 3 times; the wait doubles every time (`0.05 s × 2^n`) with a small random jitter. A call that still fails after the last retry counts as a permanent failure.

In [8]:
def call_llm_with_retry(messages, ticket_id):
    global RETRY_FAILURE_COUNT, PERMANENT_FAILURE_COUNT

    for retry_number in range(MAX_RETRIES + 1):
        try:
            response = mock_llm.chat(messages)
            return response

        except mock_llm.TransientLLMError:
            RETRY_FAILURE_COUNT += 1

            if retry_number == MAX_RETRIES:
                PERMANENT_FAILURE_COUNT += 1
                raise

            wait_time = (
                BASE_DELAY
                * (2 ** retry_number)
                * random.uniform(0.9, 1.15)
            )

            print(
                f"Ticket {ticket_id} | "
                f"Retry {retry_number} | "
                f"Waiting {wait_time:.4f} seconds"
            )

            time.sleep(wait_time)

## Task 5. Output validation
1. Try to parse JSON and accept the category only if it is on the allow-list.
2. If the reply is not JSON, accept it only when **exactly one** allowed category word appears (`re.findall` on whole words).
3. Everything else is flagged as `flagged_invalid_output` and sent to a human instead of being guessed.

In [9]:
def validate_response(response_text):

    # Try JSON first
    try:
        result = json.loads(response_text)

        category = result.get("category")

        if category in ALLOWED_CATEGORIES:
            return category, "classified_json"

        return None, "flagged_invalid_output"

    except (json.JSONDecodeError, AttributeError):
        pass

    # Keyword extraction for non-JSON replies
    words = re.findall(
        r"[a-z]+",
        response_text.lower()
    )

    matched_categories = [
        category
        for category in ALLOWED_CATEGORIES
        if category in words
    ]

    if len(matched_categories) == 1:
        return matched_categories[0], "classified_keyword"

    return None, "flagged_invalid_output"


print("\n========== VALIDATION TEST ==========")

print(
    validate_response(
        '{"category": "billing", "reason": "wrong invoice"}'
    )
)

print(
    validate_response(
        "The customer appears to have a billing-related problem."
    )
)

print(
    validate_response(
        '{"category": "shipping", "reason": "parcel"}'
    )
)

print(
    validate_response(
        "billing and fraud issue"
    )
)


========== VALIDATION TEST ==========
('billing', 'classified_json')
('billing', 'classified_keyword')
(None, 'flagged_invalid_output')
(None, 'flagged_invalid_output')


## Task 6. Full pipeline, zero-shot run and confusion matrix
`run_pipeline` applies guardrails → token budget → redaction check → model call with retry → validation, and records one row per ticket. `mock_llm.reset()` and the retry counters are reset once at the start of every complete run.

In [10]:
def run_pipeline(system_prompt, token_budget=None):
    global RETRY_FAILURE_COUNT, PERMANENT_FAILURE_COUNT

    # Reset mock and retry counters once before each complete run
    mock_llm.reset()
    RETRY_FAILURE_COUNT = 0
    PERMANENT_FAILURE_COUNT = 0

    results = []
    running_tokens = 0

    for _, ticket in tickets.iterrows():

        ticket_id = ticket["ticket_id"]
        customer_id = ticket["customer_id"]
        original_text = ticket["text"]
        redacted_text = ticket["redacted_text"]

        category = None
        status = None
        tokens_used = 0

        # --------------------------
        # INPUT GUARDRAILS
        # --------------------------

        if original_text.strip() == "":
            status = "skipped_empty"

        elif mock_llm.est_tokens(original_text) > MAX_INPUT_TOKENS:
            status = "flagged_long"

        # --------------------------
        # TOKEN BUDGET
        # --------------------------

        elif token_budget is not None and running_tokens >= token_budget:
            status = "budget_exhausted"

        # --------------------------
        # MODEL CALL
        # --------------------------

        else:

            # Verify that redacted text contains no PII
            assert not re.search(EMAIL_PATTERN, redacted_text)
            assert not re.search(PHONE_PATTERN, redacted_text)
            assert not re.search(ORDER_PATTERN, redacted_text)

            messages = [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": redacted_text
                }
            ]

            response = call_llm_with_retry(
                messages,
                ticket_id
            )

            tokens_used = response.usage.total_tokens
            running_tokens += tokens_used

            response_text = response.choices[0].message.content

            category, status = validate_response(
                response_text
            )

        # --------------------------
        # SAVE RESULT
        # --------------------------

        results.append({
            "ticket_id": ticket_id,
            "customer_id": customer_id,
            "status": status,
            "category": category,
            "tokens_used": tokens_used,
            "true_category": ticket["true_category"]
        })

    results_df = pd.DataFrame(results)

    print("\n========== PIPELINE RESULTS ==========")
    print(results_df["status"].value_counts())

    print("\nTotal tickets:", len(results_df))
    print("Total tokens:", running_tokens)

    classified = results_df[
        results_df["status"].isin([
            "classified_json",
            "classified_keyword"
        ])
    ]

    print("Classified tickets:", len(classified))

    if len(classified) > 0:
        accuracy = (
            classified["category"]
            == classified["true_category"]
        ).mean()

        tokens_per_classified = running_tokens / len(classified)

        print("Accuracy:", accuracy)
        print("Tokens per classified ticket:", round(tokens_per_classified, 1))
    else:
        print("Accuracy: N/A")
        print("Tokens per classified ticket: N/A")

    print(f"Total transient failures: {RETRY_FAILURE_COUNT}")
    print(f"Total permanent failures: {PERMANENT_FAILURE_COUNT}")

    return results_df

# ==========================================
# ZERO-SHOT FULL RUN
# ==========================================

zero_shot_results = run_pipeline(
    ZERO_SHOT_PROMPT
)

print("\n========== ZERO-SHOT STATUS COUNTS ==========")
print(zero_shot_results["status"].value_counts())

Ticket T0002 | Retry 0 | Waiting 0.0508 seconds
Ticket T0116 | Retry 0 | Waiting 0.0495 seconds
Ticket T0153 | Retry 0 | Waiting 0.0566 seconds
Ticket T0039 | Retry 0 | Waiting 0.0530 seconds
Ticket T0152 | Retry 0 | Waiting 0.0567 seconds
Ticket T0043 | Retry 0 | Waiting 0.0452 seconds
Ticket T0111 | Retry 0 | Waiting 0.0512 seconds
Ticket T0155 | Retry 0 | Waiting 0.0549 seconds
Ticket T0100 | Retry 0 | Waiting 0.0539 seconds
Ticket T0068 | Retry 0 | Waiting 0.0470 seconds
Ticket T0135 | Retry 0 | Waiting 0.0536 seconds
Ticket T0009 | Retry 0 | Waiting 0.0475 seconds
Ticket T0067 | Retry 0 | Waiting 0.0461 seconds
Ticket T0106 | Retry 0 | Waiting 0.0500 seconds
Ticket T0097 | Retry 0 | Waiting 0.0463 seconds
Ticket T0003 | Retry 0 | Waiting 0.0551 seconds
Ticket T0019 | Retry 0 | Waiting 0.0514 seconds
Ticket T0086 | Retry 0 | Waiting 0.0483 seconds
Ticket T0065 | Retry 0 | Waiting 0.0464 seconds

========== PIPELINE RESULTS ==========
status
classified_json           135
classified_

In [11]:
# ==========================================
# TASK 6: ZERO-SHOT CONFUSION MATRIX
# ==========================================

classified_zero = zero_shot_results[
    zero_shot_results["status"].isin([
        "classified_json",
        "classified_keyword"
    ])
]

confusion_df = pd.crosstab(
    classified_zero["true_category"],
    classified_zero["category"]
)

confusion_df = confusion_df.reindex(
    index=["billing", "delivery", "technical", "fraud"],
    columns=["billing", "delivery", "technical", "fraud"],
    fill_value=0
)

print("\n========== ZERO-SHOT CONFUSION MATRIX ==========")


========== ZERO-SHOT CONFUSION MATRIX ==========


## Task 6b. Few-shot run
The same pipeline with the few-shot prompt, to compare accuracy and tokens per classified ticket with the zero-shot run.

In [12]:
few_shot_results = run_pipeline(FEW_SHOT_PROMPT)
# ==========================================
# FEW-SHOT METRICS
# ==========================================

classified_few = few_shot_results[
    few_shot_results["status"].isin([
        "classified_json",
        "classified_keyword"
    ])
]

few_accuracy = (
    classified_few["category"]
    == classified_few["true_category"]
).mean()

few_tokens = few_shot_results["tokens_used"].sum()

print("\n========== FEW-SHOT METRICS ==========")
print("Classified tickets:", len(classified_few))
print("Accuracy:", few_accuracy)
print("Total tokens:", few_tokens)
print("Tokens per classified ticket:", few_tokens / len(classified_few))

Ticket T0002 | Retry 0 | Waiting 0.0514 seconds


Ticket T0116 | Retry 0 | Waiting 0.0563 seconds
Ticket T0153 | Retry 0 | Waiting 0.0557 seconds
Ticket T0039 | Retry 0 | Waiting 0.0513 seconds
Ticket T0152 | Retry 0 | Waiting 0.0515 seconds
Ticket T0043 | Retry 0 | Waiting 0.0570 seconds
Ticket T0111 | Retry 0 | Waiting 0.0510 seconds
Ticket T0155 | Retry 0 | Waiting 0.0503 seconds
Ticket T0100 | Retry 0 | Waiting 0.0457 seconds
Ticket T0068 | Retry 0 | Waiting 0.0551 seconds
Ticket T0135 | Retry 0 | Waiting 0.0559 seconds
Ticket T0009 | Retry 0 | Waiting 0.0510 seconds
Ticket T0067 | Retry 0 | Waiting 0.0561 seconds
Ticket T0106 | Retry 0 | Waiting 0.0451 seconds
Ticket T0097 | Retry 0 | Waiting 0.0465 seconds
Ticket T0003 | Retry 0 | Waiting 0.0531 seconds
Ticket T0019 | Retry 0 | Waiting 0.0522 seconds
Ticket T0086 | Retry 0 | Waiting 0.0551 seconds
Ticket T0065 | Retry 0 | Waiting 0.0516 seconds

========== PIPELINE RESULTS ==========
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9

## Task 7a. Token budget demo
With a small budget (5,000 tokens) the pipeline stops calling the model once the budget is used up and marks the remaining tickets `budget_exhausted` for human review.

In [13]:
os.makedirs("outputs", exist_ok=True)

TOKEN_BUDGET = 5000

budget_results = run_pipeline(
    ZERO_SHOT_PROMPT,
    token_budget=TOKEN_BUDGET
)

budget_results.to_csv(
    "outputs/triage_budget_demo.csv",
    index=False
)

budget_exhausted_count = (
    budget_results["status"] == "budget_exhausted"
).sum()

print("\n========== BUDGET DEMO ==========")
print("Token budget:", TOKEN_BUDGET)
print("Budget exhausted tickets:", budget_exhausted_count)
print("Saved: outputs/triage_budget_demo.csv")

Ticket T0002 | Retry 0 | Waiting 0.0530 seconds
Ticket T0116 | Retry 0 | Waiting 0.0519 seconds
Ticket T0153 | Retry 0 | Waiting 0.0557 seconds
Ticket T0039 | Retry 0 | Waiting 0.0515 seconds
Ticket T0152 | Retry 0 | Waiting 0.0453 seconds
Ticket T0043 | Retry 0 | Waiting 0.0505 seconds
Ticket T0111 | Retry 0 | Waiting 0.0527 seconds
Ticket T0155 | Retry 0 | Waiting 0.0475 seconds
Ticket T0100 | Retry 0 | Waiting 0.0512 seconds

========== PIPELINE RESULTS ==========
status
budget_exhausted          80
classified_json           67
classified_keyword         5
skipped_empty              4
flagged_long               4
flagged_invalid_output     4
Name: count, dtype: int64

Total tickets: 164
Total tokens: 5050
Classified tickets: 72
Accuracy: 0.8333333333333334
Tokens per classified ticket: 70.1
Total transient failures: 9
Total permanent failures: 0

========== BUDGET DEMO ==========
Token budget: 5000
Budget exhausted tickets: 80
Saved: outputs/triage_budget_demo.csv


## Task 7b–7e. Final run, routing by customer segment, output file and validation
Routing rules:
- guardrail / invalid-output / budget cases → `human_review`
- category `fraud` → `security_team`, priority HIGH (fraud overrides the customer segment)
- otherwise **Premium** customers (from Part B) → `premium_desk`, priority HIGH
- everything else → `support_queue`, priority NORMAL

In [14]:
# ==========================================
# TASK 7B: FINAL FEW-SHOT RUN
# ==========================================

TOKEN_BUDGET = 100000

final_results = run_pipeline(
    FEW_SHOT_PROMPT,
    token_budget=TOKEN_BUDGET
)

Ticket T0002 | Retry 0 | Waiting 0.0486 seconds
Ticket T0116 | Retry 0 | Waiting 0.0567 seconds
Ticket T0153 | Retry 0 | Waiting 0.0465 seconds
Ticket T0039 | Retry 0 | Waiting 0.0541 seconds
Ticket T0152 | Retry 0 | Waiting 0.0471 seconds
Ticket T0043 | Retry 0 | Waiting 0.0523 seconds
Ticket T0111 | Retry 0 | Waiting 0.0573 seconds
Ticket T0155 | Retry 0 | Waiting 0.0487 seconds
Ticket T0100 | Retry 0 | Waiting 0.0517 seconds
Ticket T0068 | Retry 0 | Waiting 0.0486 seconds
Ticket T0135 | Retry 0 | Waiting 0.0453 seconds
Ticket T0009 | Retry 0 | Waiting 0.0551 seconds
Ticket T0067 | Retry 0 | Waiting 0.0526 seconds
Ticket T0106 | Retry 0 | Waiting 0.0527 seconds
Ticket T0097 | Retry 0 | Waiting 0.0552 seconds
Ticket T0003 | Retry 0 | Waiting 0.0561 seconds
Ticket T0019 | Retry 0 | Waiting 0.0545 seconds
Ticket T0086 | Retry 0 | Waiting 0.0573 seconds
Ticket T0065 | Retry 0 | Waiting 0.0462 seconds

========== PIPELINE RESULTS ==========
status
classified_json           135
classified_

In [15]:
# ==========================================
# CUSTOMER SEGMENT INFORMATION
# ==========================================

customer_segments = pd.read_csv(
    "outputs/customer_segments.csv"
)

# Keep only the required customer information
customer_segments = customer_segments[
    ["customer_id", "segment_name", "is_premium"]
]

In [16]:
# ==========================================
# TASK 7C: ROUTING
# ==========================================

final_results = final_results.merge(
    customer_segments,
    on="customer_id",
    how="left"
)

# Missing segment customers are treated as not premium
final_results["is_premium"] = (
    final_results["is_premium"].fillna(False)
)

# Default routing
final_results["route"] = "support_queue"
final_results["priority"] = "NORMAL"

# Human review statuses
human_review_statuses = [
    "skipped_empty",
    "flagged_long",
    "flagged_invalid_output",
    "budget_exhausted"
]

human_review_mask = final_results["status"].isin(
    human_review_statuses
)

final_results.loc[
    human_review_mask, "route"
] = "human_review"

# Fraud tickets go to security team
fraud_mask = (
    (final_results["category"] == "fraud")
    & (~human_review_mask)
)

final_results.loc[
    fraud_mask, "route"
] = "security_team"

final_results.loc[
    fraud_mask, "priority"
] = "HIGH"

# Premium customers go to premium desk
premium_mask = (
    final_results["is_premium"]
    & (~human_review_mask)
    & (~fraud_mask)
)

final_results.loc[
    premium_mask, "route"
] = "premium_desk"

final_results.loc[
    premium_mask, "priority"
] = "HIGH"

In [17]:
# ==========================================
# TASK 7D: SAVE FINAL OUTPUT
# ==========================================

final_results["segment_name"] = (
    final_results["segment_name"].fillna("")
)

final_results["category"] = (
    final_results["category"].fillna("")
)

final_output = final_results[
    [
        "ticket_id",
        "customer_id",
        "status",
        "category",
        "segment_name",
        "priority",
        "route",
        "tokens_used"
    ]
]

final_output.to_csv(
    "outputs/triage_results.csv",
    index=False
)

print("\n========== FINAL TRIAGE OUTPUT ==========")
print("Total tickets:", len(final_output))

print("\nRoute counts:")
print(final_output["route"].value_counts())

print("\nPriority counts:")
print(final_output["priority"].value_counts())

print("\nStatus counts:")
print(final_output["status"].value_counts())

print("\nSaved: outputs/triage_results.csv")


========== FINAL TRIAGE OUTPUT ==========
Total tickets: 164

Route counts:
route
support_queue    102
security_team     34
human_review      17
premium_desk      11
Name: count, dtype: int64

Priority counts:
priority
NORMAL    119
HIGH       45
Name: count, dtype: int64

Status counts:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
skipped_empty               4
flagged_long                4
Name: count, dtype: int64

Saved: outputs/triage_results.csv


In [18]:
# ==========================================
# TASK 7E: FINAL VALIDATION
# ==========================================

assert len(final_output) == 164

assert final_output["ticket_id"].is_unique

assert final_output.loc[
    final_output["status"].isin(human_review_statuses),
    "category"
].eq("").all()

print("\nFINAL OUTPUT VALIDATION PASSED")


FINAL OUTPUT VALIDATION PASSED


# PART D — Policy RAG Assistant and Refund Agent

## Task 1. Chunk the policy documents
Every policy `.txt` file is split into 40-word chunks with a 10-word overlap, so a sentence cut at a chunk boundary still appears whole in the neighbouring chunk.

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
# ==========================================
# TASK 1: CHUNK POLICY DOCUMENTS
# ==========================================

def chunk_documents(chunk_size=40, overlap=10):

    chunks = []

    policy_folder = "data/policies"

    for filename in sorted(os.listdir(policy_folder)):

        if not filename.endswith(".txt"):
            continue

        filepath = os.path.join(policy_folder, filename)

        with open(filepath, "r", encoding="utf-8") as file:
            text = file.read()

        words = text.split()

        start = 0

        while start < len(words):

            end = start + chunk_size

            chunk_text = " ".join(words[start:end])

            chunks.append({
                "source": filename,
                "text": chunk_text
            })

            if end >= len(words):
                break

            start += chunk_size - overlap

    return chunks


# Execute chunking
chunks = chunk_documents()

print("\n========== POLICY CHUNKING ==========")
print("Total chunks:", len(chunks))

print(pd.Series([c["source"] for c in chunks]).value_counts())


========== POLICY CHUNKING ==========
Total chunks: 15
refund_policy.txt        4
delivery_policy.txt      3
escalation_policy.txt    3
payment_policy.txt       3
privacy_policy.txt       2
Name: count, dtype: int64


## Task 2. TF-IDF vectorisation and retrieval
Chunks are turned into TF-IDF vectors; a question is vectorised with the **same fitted** vectorizer and the top-k chunks are ranked by cosine similarity.

In [21]:
# ==========================================
# TASK 2: TF-IDF VECTORISATION AND RETRIEVAL
# ==========================================

# Extract text from all chunks
chunk_texts = [chunk["text"] for chunk in chunks]

# Fit TF-IDF on policy chunks
vectorizer = TfidfVectorizer()
chunk_vectors = vectorizer.fit_transform(chunk_texts)


def retrieve(query, k=3):

    # Transform query using the same fitted vectorizer
    query_vector = vectorizer.transform([query])

    # Calculate cosine similarity
    scores = cosine_similarity(query_vector, chunk_vectors)[0]

    # Get top k chunk indexes
    top_indices = np.argsort(scores)[::-1][:k]

    results = []

    for index in top_indices:
        results.append({
            "source": chunks[index]["source"],
            "score": float(scores[index]),
            "text": chunks[index]["text"]
        })

    return results


# Test retrieval
test_results = retrieve("refund policy late delivery")

print("\n========== RETRIEVAL TEST ==========")

for result in test_results:
    print("Source:", result["source"])
    print("Score:", round(result["score"], 4))
    print("Text:", result["text"][:100], "...")
    print()


========== RETRIEVAL TEST ==========
Source: refund_policy.txt
Score: 0.2719
Text: order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. ...

Source: refund_policy.txt
Score: 0.2514
Text: Refund Policy. A refund can be requested only for orders whose status is completed. Orders with stat ...

Source: refund_policy.txt
Score: 0.2454
Text: late-delivery refund is paid. Refunds up to INR 1000 are approved automatically by the support syste ...



## Task 3. Grounded policy assistant
The top-3 chunks are placed in the prompt as `CONTEXT:`, followed by the `QUESTION:`. The system prompt tells the model to answer only from the context and to use the exact fallback sentence when the answer is not there.

In [22]:
POLICY_SYSTEM_PROMPT = """
You are a POLICY ASSISTANT.

Answer questions ONLY using the supplied policy context.
Do not use outside knowledge or invent information.
If the answer is not available in the context,
respond exactly:
I could not find that information in the policy documents.
"""


def answer(question):

    # Retrieve top 3 relevant policy chunks
    retrieved = retrieve(question, k=3)

    # Build context
    context_parts = []

    for item in retrieved:
        context_parts.append(
            f"[{item['source']}]\n{item['text']}"
        )

    context = "\n".join(context_parts)

    # User message format
    user_message = (
        f"CONTEXT:\n{context}\n"
        f"QUESTION:\n{question}"
    )

    messages = [
        {
            "role": "system",
            "content": POLICY_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    # Call mock LLM
    response = mock_llm.chat(messages)

    final_answer = response.choices[0].message.content

    # Return answer and sources
    sources = [item["source"] for item in retrieved]

    return {
        "answer": final_answer,
        "sources": sources
    }


# Test policy assistant
print("\n========== POLICY ASSISTANT TEST ==========")

result = answer("What is the refund policy for late delivery?")

print("Answer:", result["answer"])


========== POLICY ASSISTANT TEST ==========
Answer: Refunds up to INR 1000 are
 
Refund Policy.


## Task 4. Retrieval and grounding evaluation
For every question in `policy_eval.csv` check (a) whether the expected document was retrieved, (b) whether the expected fact appears in the answer, and (c) for unanswerable questions (`expected_doc = NONE`) whether the assistant used the fallback.

In [23]:
def evaluate_policy():

    evaluation = pd.read_csv(
        "data/raw/policy_eval.csv",
        keep_default_na=False
    )

    results = []

    for _, row in evaluation.iterrows():

        question = row["question"]
        expected_doc = row["expected_doc"]
        expected_fact = row["expected_fact"]

        retrieved = retrieve(question, k=3)
        result = answer(question)

        retrieved_docs = [
            item["source"] for item in retrieved
        ]

        # Empty expected_doc means unanswerable
        answerable = expected_doc.strip().upper() != "NONE"

        doc_found = (
            expected_doc in retrieved_docs
            if answerable else False
        )

        fact_found = (
            expected_fact in result["answer"]
            if answerable else False
        )

        fallback_found = result["answer"].startswith(
            "I could not find"
        )

        results.append({
            "question": question,
            "expected_doc": expected_doc,
            "answerable": answerable,
            "doc_found": doc_found,
            "fact_found": fact_found,
            "fallback_found": fallback_found
        })

    results_df = pd.DataFrame(results)

    answerable_results = results_df[
        results_df["answerable"]
    ]

    unanswerable_results = results_df[
        ~results_df["answerable"]
    ]

    print("\n========== POLICY EVALUATION ==========")

    print(
        "Expected document retrieved:",
        answerable_results["doc_found"].sum(),
        "/",
        len(answerable_results)
    )

    print(
        "Expected fact found:",
        answerable_results["fact_found"].sum(),
        "/",
        len(answerable_results)
    )

    print(
        "Fallback answers:",
        unanswerable_results["fallback_found"].sum(),
        "/",
        len(unanswerable_results)
    )

    return results_df


evaluation_results = evaluate_policy()


========== POLICY EVALUATION ==========
Expected document retrieved: 12 / 12
Expected fact found: 12 / 12
Fallback answers: 2 / 2


### Task 4b. Effect of chunk overlap
Re-run the same evaluation with **zero overlap** and list the answerable questions that now fail. Then restore the original overlapping chunks and vectorizer.

In [24]:
# Create chunks without overlap
chunks_zero_overlap = chunk_documents(
    chunk_size=40,
    overlap=0
)

# Temporarily replace original chunks
original_chunks = chunks
chunks = chunks_zero_overlap

# Fit a new TF-IDF vectorizer
chunk_texts = [chunk["text"] for chunk in chunks]

vectorizer = TfidfVectorizer()
chunk_vectors = vectorizer.fit_transform(chunk_texts)

# Evaluate again
zero_overlap_results = evaluate_policy()

failed_zero_overlap = zero_overlap_results[
    zero_overlap_results["answerable"] & ~zero_overlap_results["fact_found"]
]
print("\nAnswerable questions that failed with zero overlap:")
for question in failed_zero_overlap["question"]:
    print("-", question)

print("\n========== ZERO OVERLAP CHUNKING ==========")
print("Total chunks:", len(chunks_zero_overlap))

# Restore original chunks and vectorizer
chunks = original_chunks

chunk_texts = [chunk["text"] for chunk in chunks]
vectorizer = TfidfVectorizer()
chunk_vectors = vectorizer.fit_transform(chunk_texts)

print("Original overlapping chunks restored.")


========== POLICY EVALUATION ==========
Expected document retrieved: 12 / 12
Expected fact found: 10 / 12
Fallback answers: 2 / 2

Answerable questions that failed with zero overlap:
- What delivery time makes a completed order eligible for a full refund?
- How soon is money returned when a payment fails but is deducted?

========== ZERO OVERLAP CHUNKING ==========
Total chunks: 13
Original overlapping chunks restored.


## Task 5. Refund tools
Five tools the agent can call: `get_order` (SQLite lookup), `search_policy` (RAG), `calculate_refund` (full refund only when delivery took more than 45 minutes), `issue_refund` and `escalate_to_human` (both write to the ledger `outputs/refund_ledger.jsonl`). `TOOLS_SPEC` describes them in JSON-Schema style and `TOOL_MAP` connects the names to the Python functions.

In [25]:
from datetime import datetime
# Tool 1: Get order
def get_order(order_id):

    conn = sqlite3.connect("data/instamart.db")

    query = """
        SELECT order_id, status, order_amount, delivery_minutes
        FROM orders
        WHERE order_id = ?
    """

    result = pd.read_sql_query(
        query,
        conn,
        params=(order_id,)
    )

    conn.close()

    if result.empty:
        return {"error": "order not found"}

    return result.iloc[0].to_dict()


# Tool 2: Search policy
def search_policy(query):

    results = retrieve(query, k=3)

    return {"results": results}


# Tool 3: Calculate refund
def calculate_refund(order_amount, delivery_minutes):

    if delivery_minutes > 45:
        refund = float(order_amount)
    else:
        refund = 0.0

    return {"refund_inr": refund}


# Helper: Append ledger entry
def write_ledger(tool_name, order_id, refund_inr):

    os.makedirs("outputs", exist_ok=True)

    entry = {
        "timestamp": datetime.now().isoformat(),
        "tool_name": tool_name,
        "order_id": order_id,
        "refund_inr": refund_inr
    }

    with open(
        "outputs/refund_ledger.jsonl",
        "a",
        encoding="utf-8"
    ) as file:

        file.write(json.dumps(entry) + "\n")


# Tool 4: Issue refund
def issue_refund(order_id, refund_inr):

    write_ledger(
        "issue_refund",
        order_id,
        refund_inr
    )

    return {"refund_inr": refund_inr}


# Tool 5: Escalate to human
def escalate_to_human(order_id, refund_inr):

    write_ledger(
        "escalate_to_human",
        order_id,
        refund_inr
    )

    return {"refund_inr": refund_inr}


# Tool specifications in JSON-Schema style
TOOLS_SPEC = [
    {
        "name": "get_order",
        "description": "Retrieve order details using its order ID.",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"]
        }
    },
    {
        "name": "search_policy",
        "description": "Retrieve relevant refund policy documents.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"]
        }
    },
    {
        "name": "calculate_refund",
        "description": "Calculate refund amount using delivery time.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_amount": {"type": "number"},
                "delivery_minutes": {"type": "number"}
            },
            "required": ["order_amount", "delivery_minutes"]
        }
    },
    {
        "name": "issue_refund",
        "description": "Record an automatically issued refund.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "refund_inr": {"type": "number"}
            },
            "required": ["order_id", "refund_inr"]
        }
    },
    {
        "name": "escalate_to_human",
        "description": "Record a refund requiring human approval.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "refund_inr": {"type": "number"}
            },
            "required": ["order_id", "refund_inr"]
        }
    }
]

# Tool map
TOOL_MAP = {
    "get_order": get_order,
    "search_policy": search_policy,
    "calculate_refund": calculate_refund,
    "issue_refund": issue_refund,
    "escalate_to_human": escalate_to_human
}


print("\n========== REFUND TOOL VERIFICATION ==========")

print("Tools:", list(TOOL_MAP.keys()))

print("Refund for 60 minutes:",
      calculate_refund(128.25, 60))

print("Refund for 30 minutes:",
      calculate_refund(128.25, 30))


========== REFUND TOOL VERIFICATION ==========
Tools: ['get_order', 'search_policy', 'calculate_refund', 'issue_refund', 'escalate_to_human']
Refund for 60 minutes: {'refund_inr': 128.25}
Refund for 30 minutes: {'refund_inr': 0.0}


## Task 6. The refund agent loop (with safety limits and checkpoints)
The agent asks the mock model what to do, executes the requested tool, appends the observation and repeats. It stops safely on:
- a **final answer** (`REFUND`, `ESCALATE`, `DENIED`, `NOT_FOUND` …),
- `max_iterations` → `FALLBACK_MAX_ITERATIONS`,
- `token_limit` → `FALLBACK_TOKEN_LIMIT`,
- `time_limit_seconds` → `FALLBACK_TIMEOUT`.

After every tool observation a checkpoint is saved in `state/<run_id>.json`, so a crashed run can be resumed by `resume_agent` **without repeating tools that already ran**.

In [26]:
def save_checkpoint(run_id, scenario_id, messages,
                   tokens_used, steps, tools_executed,
                   step_number):

    os.makedirs("state", exist_ok=True)

    checkpoint = {
        "run_id": run_id,
        "scenario_id": scenario_id,
        "messages": messages,
        "tokens_used": tokens_used,
        "steps": steps,
        "tools_executed": tools_executed,
        "step_number": step_number
    }

    checkpoint_path = f"state/{run_id}.json"

    with open(checkpoint_path, "w", encoding="utf-8") as file:
        json.dump(checkpoint, file, indent=4)


def run_agent(
    user_message,
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10,
    scenario_id=None,
    run_id=None,
    crash_after_tool=None
):

    if run_id is None:
        run_id = str(uuid.uuid4())

    messages = [
        {
            "role": "system",
            "content": "AGENT: You are an Instamart refund agent. Use tools to verify orders, check policies, calculate refunds, and process or escalate refunds."
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    steps = []
    tools_executed = []
    llm_calls = 0
    tokens_used = 0
    start_time = time.monotonic()

    status = None
    final_answer = None

    for iteration in range(max_iterations):

        # Timeout check
        if time.monotonic() - start_time >= time_limit_seconds:
            status = "FALLBACK_TIMEOUT"
            final_answer = (
                "FALLBACK_TIMEOUT: Time limit reached. "
                "Please review manually."
            )
            break

        response = mock_llm.chat(messages, tools=TOOLS_SPEC)

        llm_calls += 1
        tokens_used += response.usage.total_tokens

        assistant_message = response.choices[0].message

        # Token limit check
        if tokens_used > token_limit:
            status = "FALLBACK_TOKEN_LIMIT"
            final_answer = (
                "FALLBACK_TOKEN_LIMIT: Token limit exceeded. "
                "Please review manually."
            )
            break

        tool_calls = assistant_message.tool_calls

        # Final answer
        if not tool_calls:
            final_answer = assistant_message.content
            status = final_answer.split(":")[0].strip()
            break

        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {
                    "id": call.id,
                    "name": call.name,
                    "arguments": call.arguments
                }
                for call in tool_calls
            ]
        })

        for call in tool_calls:

            arguments = call.arguments

            if isinstance(arguments, str):
                arguments = json.loads(arguments)

            tool_name = call.name
            observation = TOOL_MAP[tool_name](**arguments)

            steps.append({
                "tool": tool_name,
                "arguments": arguments,
                "observation": observation
            })

            tools_executed.append(tool_name)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": tool_name,
                "content": json.dumps(observation)
            })

            # Save checkpoint AFTER tool observation
            save_checkpoint(
                run_id,
                scenario_id,
                messages,
                tokens_used,
                steps,
                tools_executed,
                len(steps)
            )

            # Simulate crash after requested tool execution
            if crash_after_tool is not None:
                if len(tools_executed) == crash_after_tool:
                    raise RuntimeError("SIMULATED CRASH")

    if final_answer is None and status is None:
        status = "FALLBACK_MAX_ITERATIONS"
        final_answer = (
            "FALLBACK_MAX_ITERATIONS: Agent reached the maximum "
            "number of iterations. Please review manually."
        )

    return {
        "run_id": run_id,
        "status": status,
        "final_answer": final_answer,
        "steps": steps,
        "llm_calls": llm_calls,
        "tokens_used": tokens_used,
        "tools_executed": tools_executed
    }

In [27]:
# ============================================================
# RESUME AGENT
# ============================================================

def resume_agent(
    run_id,
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10
):

    checkpoint_path = f"state/{run_id}.json"

    with open(checkpoint_path, "r", encoding="utf-8") as file:
        checkpoint = json.load(file)

    messages = checkpoint["messages"]
    scenario_id = checkpoint["scenario_id"]
    tokens_used = checkpoint["tokens_used"]
    steps = checkpoint["steps"]
    tools_executed = checkpoint["tools_executed"]

    start_time = time.monotonic()

    status = None
    final_answer = None
    llm_calls = 0

    for iteration in range(max_iterations):

        if time.monotonic() - start_time >= time_limit_seconds:
            status = "FALLBACK_TIMEOUT"
            final_answer = (
                "FALLBACK_TIMEOUT: Time limit reached. "
                "Please review manually."
            )
            break

        response = mock_llm.chat(messages, tools=TOOLS_SPEC)

        llm_calls += 1
        tokens_used += response.usage.total_tokens

        assistant_message = response.choices[0].message

        if tokens_used > token_limit:
            status = "FALLBACK_TOKEN_LIMIT"
            final_answer = (
                "FALLBACK_TOKEN_LIMIT: Token limit exceeded. "
                "Please review manually."
            )
            break

        tool_calls = assistant_message.tool_calls

        if not tool_calls:
            final_answer = assistant_message.content
            status = final_answer.split(":")[0].strip()
            break

        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {
                    "id": call.id,
                    "name": call.name,
                    "arguments": call.arguments
                }
                for call in tool_calls
            ]
        })

        for call in tool_calls:

            arguments = call.arguments

            if isinstance(arguments, str):
                arguments = json.loads(arguments)

            tool_name = call.name
            observation = TOOL_MAP[tool_name](**arguments)

            steps.append({
                "tool": tool_name,
                "arguments": arguments,
                "observation": observation
            })

            tools_executed.append(tool_name)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": tool_name,
                "content": json.dumps(observation)
            })

            save_checkpoint(
                run_id,
                scenario_id,
                messages,
                tokens_used,
                steps,
                tools_executed,
                len(steps)
            )

    if final_answer is None and status is None:
        status = "FALLBACK_MAX_ITERATIONS"
        final_answer = (
            "FALLBACK_MAX_ITERATIONS: Agent reached the maximum "
            "number of iterations. Please review manually."
        )

    return {
        "run_id": run_id,
        "status": status,
        "final_answer": final_answer,
        "steps": steps,
        "llm_calls": llm_calls,
        "tokens_used": tokens_used,
        "tools_executed": tools_executed
    }

### Run all 7 scenarios
Each scenario is checked against its expected status and refund amount in `agent_scenarios.csv` (scenario S7 is the `[LOOP]` case and must end in a fallback). Results are written to `outputs/agent_runs.jsonl`.

In [28]:
scenarios = pd.read_csv("data/raw/agent_scenarios.csv")

os.makedirs("outputs", exist_ok=True)

# Start each full run with a clean ledger and clean checkpoints
open("outputs/refund_ledger.jsonl", "w", encoding="utf-8").close()
os.makedirs("state", exist_ok=True)
for old_checkpoint in os.listdir("state"):
    if old_checkpoint.endswith(".json"):
        os.remove(os.path.join("state", old_checkpoint))

agent_results = []

# Reset mock before the independent scenario batch
mock_llm.reset()

with open("outputs/agent_runs.jsonl", "w", encoding="utf-8") as file:

    for _, scenario in scenarios.iterrows():

        scenario_id = scenario["scenario_id"]

        # Identify the user-message column
        user_message = scenario.get(
            "user_message",
            scenario.get("request", scenario.get("message", None))
        )

        if pd.isna(user_message):
            raise ValueError(
                f"User message missing for scenario {scenario_id}"
            )

        result = run_agent(
            user_message=str(user_message),
            scenario_id=scenario_id
        )

        # Save scenario result
        record = {
            "scenario_id": scenario_id,
            "status": result["status"],
            "final_answer": result["final_answer"],
            "llm_calls": result["llm_calls"],
            "tokens_used": result["tokens_used"],
            "steps": result["steps"]
        }

        # DOCX Task 6 verification: expected status and refund amount
        expected_status = str(scenario["expected_outcome"])
        if scenario_id == "S7":
            assert str(result["status"]).startswith("FALLBACK"), (
                f"S7 must fall back, got {result['status']}"
            )
        else:
            assert result["status"] == expected_status, (
                f"{scenario_id}: expected status {expected_status}, "
                f"got {result['status']}"
            )

        if scenario_id in {"S1", "S2", "S3", "S4", "S5", "S6"}:
            refund_observations = [
                step["observation"]["refund_inr"]
                for step in result["steps"]
                if step["tool"] == "calculate_refund"
                and "refund_inr" in step.get("observation", {})
            ]
            expected_refund = float(scenario["expected_refund_inr"])
            if refund_observations:
                actual_refund = float(refund_observations[-1])
            else:
                # NOT_FOUND scenarios never reach calculate_refund; no refund is issued.
                actual_refund = 0.0
            assert abs(actual_refund - expected_refund) <= 0.01, (
                f"{scenario_id}: expected refund INR {expected_refund:.2f}, "
                f"got INR {actual_refund:.2f}"
            )

        file.write(json.dumps(record) + "\n")

        agent_results.append(record)

        print(f"\nScenario: {scenario_id}")
        print("Status:", result["status"])
        print("Final answer:", result["final_answer"])
        print("LLM calls:", result["llm_calls"])
        print("Tokens:", result["tokens_used"])
        print("Tools:", [step["tool"] for step in result["steps"]])


# ============================================================
# AGENT SUMMARY
# ============================================================

agent_results_df = pd.DataFrame(agent_results)

print("\n========== AGENT SCENARIO RESULTS ==========")

print(
    agent_results_df[
        ["scenario_id", "status", "llm_calls", "tokens_used"]
    ].to_string(index=False)
)

print("\nSaved: outputs/agent_runs.jsonl")


Scenario: S1
Status: REFUND
Final answer: REFUND: INR 128.25 issued.
LLM calls: 5
Tokens: 972
Tools: ['get_order', 'search_policy', 'calculate_refund', 'issue_refund']

Scenario: S2
Status: REFUND
Final answer: REFUND: INR 135.0 issued.
LLM calls: 5
Tokens: 972
Tools: ['get_order', 'search_policy', 'calculate_refund', 'issue_refund']

Scenario: S3
Status: ESCALATE
Final answer: ESCALATE: refund above INR 1000 sent to a human lead.
LLM calls: 5
Tokens: 959
Tools: ['get_order', 'search_policy', 'calculate_refund', 'escalate_to_human']

Scenario: S4
Status: DENIED
Final answer: DENIED: delivery was within 45 minutes.
LLM calls: 4
Tokens: 665
Tools: ['get_order', 'search_policy', 'calculate_refund']

Scenario: S5
Status: DENIED
Final answer: DENIED: only completed orders can be refunded.
LLM calls: 3
Tokens: 382
Tools: ['get_order', 'search_policy']

Scenario: S6
Status: NOT_FOUND
Final answer: NOT_FOUND: no such order.
LLM calls: 2
Tokens: 102
Tools: ['get_order']

Scenario: S7
Status: F

## Task 7. Stop-safely demonstrations
The same request run with (a) a deliberately tiny token limit and (b) a zero-second time limit. Both must end in a fallback status instead of running on.

In [29]:
print("\n========== TOKEN LIMIT DEMO ==========")

s1_row = scenarios.loc[
    scenarios["scenario_id"] == "S1"
].iloc[0]

# Automatically identify the user-message column
message_column = next(
    col for col in ["user_message", "request", "message"]
    if col in scenarios.columns
)

s1_message = str(s1_row[message_column])

mock_llm.reset()

token_demo = run_agent(
    user_message=s1_message,
    scenario_id="S1_TOKEN_DEMO",
    token_limit=200
)

print("Status:", token_demo["status"])
print("Final answer:", token_demo["final_answer"])
print("LLM calls:", token_demo["llm_calls"])
print("Tokens:", token_demo["tokens_used"])


========== TOKEN LIMIT DEMO ==========
Status: FALLBACK_TOKEN_LIMIT
Final answer: FALLBACK_TOKEN_LIMIT: Token limit exceeded. Please review manually.
LLM calls: 3
Tokens: 399


In [30]:
# TASK 7: TIMEOUT DEMO
# ============================================================

print("\n========== TIMEOUT DEMO ==========")

mock_llm.reset()

timeout_demo = run_agent(
    user_message=s1_message,
    scenario_id="S1_TIMEOUT_DEMO",
    time_limit_seconds=0
)

print("Status:", timeout_demo["status"])
print("Final answer:", timeout_demo["final_answer"])
print("LLM calls:", timeout_demo["llm_calls"])
print("Tokens:", timeout_demo["tokens_used"])


========== TIMEOUT DEMO ==========
Status: FALLBACK_TIMEOUT
Final answer: FALLBACK_TIMEOUT: Time limit reached. Please review manually.
LLM calls: 0
Tokens: 0


## Task 8. Crash and resume
The first run is crashed on purpose after its 2nd tool call. `resume_agent` then loads the checkpoint and continues. The tools list across the crash and the resume must show every tool **once** (no tool executed twice).

In [31]:
# ============================================================

print("\n========== CRASH AND RESUME DEMO ==========")

# Reuse s1_message identified in the token-limit demo

# Create a unique run ID
resume_run_id = "S1_RESUME_DEMO"

# Reset mock before starting this independent run
mock_llm.reset()

# First execution: simulate crash after second tool
try:
    first_run = run_agent(
        user_message=s1_message,
        scenario_id="S1",
        run_id=resume_run_id,
        crash_after_tool=2
    )

except RuntimeError as error:
    print("Simulated crash:", error)

# Resume from saved checkpoint
resumed_result = resume_agent(resume_run_id)

print("\n========== RESUMED RESULT ==========")
print("Status:", resumed_result["status"])
print("Final answer:", resumed_result["final_answer"])
print("Tools executed:", resumed_result["tools_executed"])
print("LLM calls after resume:", resumed_result["llm_calls"])
print("Tokens after resume:", resumed_result["tokens_used"])

print("\nTotal tools across crash and resume:")
print(resumed_result["tools_executed"])


========== CRASH AND RESUME DEMO ==========
Simulated crash: SIMULATED CRASH

========== RESUMED RESULT ==========
Status: REFUND
Final answer: REFUND: INR 128.25 issued.
Tools executed: ['get_order', 'search_policy', 'calculate_refund', 'issue_refund']
LLM calls after resume: 3
Tokens after resume: 972

Total tools across crash and resume:
['get_order', 'search_policy', 'calculate_refund', 'issue_refund']


## Files produced by this notebook

In [32]:
for path in ["outputs/triage_budget_demo.csv", "outputs/triage_results.csv",
             "outputs/agent_runs.jsonl", "outputs/refund_ledger.jsonl", "state"]:
    print(f"{'OK     ' if (ROOT / path).exists() else 'MISSING'} {path}")

OK      outputs/triage_budget_demo.csv
OK      outputs/triage_results.csv
OK      outputs/agent_runs.jsonl
OK      outputs/refund_ledger.jsonl
OK      state


In [33]:
from pathlib import Path

ROOT = Path.cwd().parent

ledger = ROOT / "outputs" / "refund_ledger.jsonl"

print("Ledger exists:", ledger.exists())

if ledger.exists():
    print("Ledger lines:", len(ledger.read_text().splitlines()))
else:
    print("Refund ledger file has not been created.")

Ledger exists: False
Refund ledger file has not been created.
